In [18]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [25]:
cd "drive/MyDrive/DL Project/Code"

/content/drive/.shortcut-targets-by-id/1MmKwE-uT3TRPifUfMKOSAl93BK5Kr9oe/DL Project/Code


In [24]:
ls

drive/  sample_data/


In [26]:
import os
import torch
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import numpy as np

In [28]:

# Define a custom dataset
class VariableLengthDataset(Dataset):
    def __init__(self, inputs, targets):
        """
        inputs: List of tensors of shape [N, 32] (variable N per sample)
        targets: List of tensors of shape [N] (variable N per sample)
        """
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

# Custom collate function for padding
def collate_fn(batch):
    """
    Pads sequences in a batch to the length of the longest sequence.
    """
    inputs, targets = zip(*batch)
    padded_inputs = pad_sequence(inputs, batch_first=True)  # Pads inputs to [batch_size, max_seq_len, 32]
    padded_targets = pad_sequence(targets, batch_first=True)  # Pads targets to [batch_size, max_seq_len]
    return padded_inputs, padded_targets

In [77]:
current_directory = os.getcwd()
data_dir = f'{current_directory}/data'
spectrogram_tensor_dir = f'{data_dir}/spectrogram_tensors'
song_breaks_dir = f'{data_dir}/segment_breaks'


def progress_count(current, total):
    ending = '\n' if current == total else ''
    print(f'\r{current}/{total}', end=ending)


def expand_ones(arr, n):
    arr = np.array(arr)
    length = len(arr)
    expanded = np.zeros_like(arr).astype(float)

    # Iterate through the array
    for i in range(length):
        if arr[i] == 1:
            # Ensure boundaries are handled
            # start = max(0, i - n)
            # end = min(length, i + n + 1)
            # expanded[start:end] = 1

            expanded[max(0, i - 2)] = min(1, expanded[max(0, i - 2)] + 0.1)
            expanded[max(0, i - 1)] = min(1, expanded[max(0, i - 1)] + 0.3)
            expanded[i] = 1
            expanded[min(length - 1, i + 1)] = min(1, expanded[min(length - 1, i + 1)] + 1)
            expanded[min(length - 1, i + 2)] = min(1, expanded[min(length - 1, i + 2)] + 0.8)
            expanded[min(length - 1, i + 3)] = min(1, expanded[min(length - 1, i + 3)] + 0.3)

    return expanded


def get_dataset(segment_length_ms=20, left=0, right=80):
    sequences = []
    labels = []

    song_file_names = os.listdir(f'{data_dir}/songs_20ms')
    tensor_files = filter(lambda x: x.endswith(".pt"), song_file_names)
    tensor_ids = sorted(map(lambda x: int(x[:-3]), tensor_files))[left:right]
    num_tensors = len(tensor_ids)

    for num, tensor_id in enumerate(tensor_ids):
        try:
          data = torch.load(f'{spectrogram_tensor_dir}/{tensor_id}.pt', weights_only=True)
          tensor = data['spectrogram'].T
          breaks = data['labels']
          breaks = expand_ones(breaks, 2)

          sequences.append(tensor)
          labels.append(torch.tensor(breaks, dtype=torch.float32))

          progress_count(num + 1, num_tensors)

        except FileNotFoundError:
          continue

    return VariableLengthDataset(sequences, labels)


In [59]:
import torch
import torch.nn as nn

class SequenceTransformer(nn.Module):
    def __init__(self, input_dim=32, n_heads=8, num_layers=2, hidden_dim=64):
        super(SequenceTransformer, self).__init__()

        # CNN
        self.feature_encoder = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Conv1d(hidden_dim, input_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(input_dim)
        )

        # Transformer encoder
        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=input_dim,
                nhead=n_heads,
                dim_feedforward=hidden_dim,
                activation='relu'
            ),
            num_layers=num_layers
        )

        # Linear head to project to scalar output per timestep
        self.output_layer = nn.Linear(input_dim, 1)

        # Sigmoid activation for output probabilities
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: [B, N, 32], expected input
        encoded = self.feature_encoder(x.permute(0, 2, 1))   # [B, N, 32]

        # Pass through transformer encoder
        encoded = self.transformer_encoder(encoded.permute(0, 2, 1))  # [B, N, 32]

        # Remove batch dimension
        encoded = encoded.squeeze(1)  # [N, 32]

        # Linear projection to scalar output for each timestep
        logits = self.output_layer(encoded).squeeze(-1)  # [N]

        # Sigmoid activation for probabilities
        probs = self.sigmoid(logits)  # [N]

        return probs


In [72]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim

# Training function
def train_model(model, dataloader, criterion, optimizer, num_epochs=10):
    model.train()  # Set model to training mode
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for inputs, targets in dataloader:
            # Move data to device
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass
            outputs = model(inputs)

            # Compute loss
            loss = criterion(outputs, targets)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        model_directory = f'{current_directory}/pretrained/Transformer_Spectrogram_Models'
        model_path = f'{model_directory}/model_{epoch + 1}_epoch_{epoch_loss / len(dataloader)}'
        torch.save(model.state_dict(), f'{model_path}')

        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss / len(dataloader):.4f}")


In [84]:
# get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Create dataset and dataloader
dataset = get_dataset(20, left=0, right=80)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

# Initialize model, loss function, and optimizer
model = SequenceTransformer().to(device)
model_directory = f'{current_directory}/pretrained/Transformer_Spectrogram_Models'
model.load_state_dict(torch.load(f'{model_directory}/model_20_epoch_0.36920341596765033'))
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Train the model
train_model(model, dataloader, criterion, optimizer, num_epochs=20)

cuda
80/80


<ipython-input-84-7accc63cd03a>:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'{model_directory}/model_20_epoch_0.36920341596765033'))


Epoch [1/20], Loss: 0.3664
Epoch [2/20], Loss: 0.3646
Epoch [3/20], Loss: 0.3642
Epoch [4/20], Loss: 0.3639
Epoch [5/20], Loss: 0.3641
Epoch [6/20], Loss: 0.3638
Epoch [7/20], Loss: 0.3639
Epoch [8/20], Loss: 0.3637
Epoch [9/20], Loss: 0.3635
Epoch [10/20], Loss: 0.3634
Epoch [11/20], Loss: 0.3634
Epoch [12/20], Loss: 0.3632


KeyboardInterrupt: 

In [88]:
from sklearn.metrics import confusion_matrix

model_directory = f'{current_directory}/pretrained/Transformer_Spectrogram_Models'

model = SequenceTransformer()
model.load_state_dict(torch.load(f'{model_directory}/model_10_epoch_0.38323369849536376', map_location=torch.device(device)))
model.to(device)
model.eval()

dataset = get_dataset(20, left=80, right=93)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)


cm = np.zeros((2, 2))

for inputs, targets in dataloader:
    # Move data to device
    inputs, targets = inputs.to(device), targets.to(device)
    outputs = model(inputs)

    threshold = 0.5
    outputs = outputs.detach().cpu().numpy()
    outputs = (outputs >= threshold).astype(int).flatten()
    targets = targets.cpu().numpy().astype(int).flatten()

    cm += confusion_matrix(targets, outputs)

print(cm)

<ipython-input-88-bd4c26e07d80>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'{model_directory}/model_10_epoch_0.38323369849536376', ma

13/13
[[54047.  5196.]
 [ 5484.  1497.]]


In [89]:
model_directory = f'{current_directory}/pretrained/Transformer_Spectrogram_Models'

model = SequenceTransformer()
model.load_state_dict(torch.load(f'{model_directory}/model_10_epoch_0.38323369849536376', map_location=torch.device(device)))
model.to(device)
model.eval()

dataset = get_dataset(20, left=80, right=93)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

for inputs, targets in dataloader:
    # Move data to device
    inputs, targets = inputs.to(device), targets.to(device)
    outputs = model(inputs)

    threshold = 0.5
    outputs = outputs.detach().cpu().numpy()
    outputs = (outputs >= threshold).astype(int).flatten()
    targets = targets.cpu().numpy()
    targets = (targets >= threshold).astype(int).flatten()

    for target in targets:
        print(target, end='')
    print("")
    for output in outputs:
        print(output, end='')
    print("\n\n")



/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
<ipython-input-89-953bc1ae2642>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you sta

13/13
...


